# 05 — Multi-Source Comparison

Combine all five sources (CDC, World Bank, WHO, OWID, UN IGME) into one tidy panel and use it for:

1. **Cross-source validation** — do the sources agree on the U.S. rate?
2. **U.S. in global context** — how does the U.S. rank against other countries?
3. **Long-run global trend** — the century-plus worldwide decline.

**Input**: all source APIs (auto-pulled and cached)

In [ ]:
# add the project folder to the path so we can import our code in src/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import harmonize

sns.set_theme()

In [ ]:
# stack every source into one tidy panel
panel = harmonize.load_all()
print(panel.shape)
panel.groupby('source').agg(rows=('imr','size'), first_year=('year','min'),
                            last_year=('year','max'), entities=('entity','nunique'))

In [ ]:
# 1. cross-source validation: U.S. rate by year and source (country-level sources)
us = harmonize.us_across_sources(panel)
us.tail(10).round(2)

In [ ]:
# the country-level sources track each other closely (they derive from UN IGME)
ax = us.plot(figsize=(9,5), marker='.')
ax.set_ylabel('deaths per 1,000'); ax.set_title('U.S. IMR across sources')
plt.show()

In [ ]:
# 2. U.S. in global context: rank among countries (World Bank), lower IMR = better
cy = harmonize.country_year(panel, source='World Bank')
year = 2018
rank = int(cy.loc[year].rank().loc['USA'])
print(f'USA ranks {rank} of {cy.loc[year].notna().sum()} countries in {year} (1 = lowest IMR)')
print('US IMR:', round(cy.loc[year,'USA'],2))
# the ten countries with the lowest IMR that year
cy.loc[year].nsmallest(10).round(2)

In [ ]:
# 3. long-run global trend: world median IMR over time (OWID has the longest history)
owid = panel[panel['source'] == 'Our World in Data']
world_median = owid.groupby('year')['imr'].median()
ax = world_median.plot(figsize=(9,5))
ax.set_ylabel('deaths per 1,000'); ax.set_title('World median infant mortality rate')
plt.show()

## Notes

- The four country-level sources agree almost exactly on the U.S. — they are not independent (all built on UN IGME), so agreement is expected, not corroboration.
- The U.S. ranks well outside the top 30 despite its wealth — a recurring public-health finding.
- WHO and UN IGME also carry uncertainty bounds (`imr_low`/`imr_high`) for interval estimates.